# MeChess 12/12: the classifier two-sample test (CPU notebook)

An adapter around `chessme humanlike-c2st`. Can a classifier tell **human** games from **generated** ones when it sees only the moves? Human and generated games start from the same positions (notebooks 10 and 11), both are cut at the length of the human game, and the features describe the moves from the mover's point of view (no clocks, no thinking time, no results, no model probabilities). The lower the AUC, the harder the generated games are to tell from human ones (0.5: not at all). Game-disjoint cross-validation, a linear model and a small network, intervals by resampling starts, differences between the generated sets paired over the same starts, and a control (human against human) that must give 0.5.

With `ENGINE_FEATURES = True` it is the detector-style variant (E13): Stockfish (an external download, GPL-3.0, pinned release) judges every move of the first `MAX_MOVES` half-moves after the start (`chessme humanlike-quality`: evaluation lost, error, blunder, engine's first choice), and these join the features. Cost: about 40 searches per game, minutes to an hour on four processes.

**Before you run:** add the outputs of notebook 10 (human games) and of notebook 11 run once per player (`games_frozen.jsonl`, `games_behaviour.jsonl`, ...) as inputs; *Accelerator -> None*, *Internet -> On*; **Save Version -> Save & Run All (Commit)**.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
WORKERS = 4
ENGINE_FEATURES = False                                 # True: add the engine-judged quality of each move (detector-style classifier, E13)
MAX_MOVES = "40"                                        # with ENGINE_FEATURES: half-moves after the start that are judged and compared (40 = the 20 moves of the forced-choice stimuli); "" = whole game
STOCKFISH_URL = "https://github.com/official-stockfish/Stockfish/releases/download/sf_18/stockfish-ubuntu-x86-64-avx2.tar"
INPUT_DIR = ""                                          # the output folders of notebooks 10 and 11; empty: search /kaggle/input

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "chess==1.11.2", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
root = INPUT_DIR or "/kaggle/input"
def find(name):
    return next(iter(sorted(glob.glob(root + "/**/" + name, recursive=True))), None)
HUMAN = find("games_human.jsonl")
GENERATED = {name: find(f"games_{name}.jsonl") for name in ("frozen", "likelihood", "behaviour")}
GENERATED = {k: v for k, v in GENERATED.items() if v}
print(HUMAN, GENERATED, sep="\n")
if not HUMAN or not GENERATED:
    raise SystemExit("Add the outputs of notebook 10 (games_human.jsonl) and notebook 11 (games_<player>.jsonl) as inputs.")
os.makedirs(OUT, exist_ok=True)

## Engine-judged move quality (only with `ENGINE_FEATURES = True`)


In [ ]:
EXTRA, ENGINE = [], None
if ENGINE_FEATURES:
    os.makedirs("/tmp/sf", exist_ok=True)
    sh("curl", "-fsSL", "-o", "/tmp/sf/sf.tar", STOCKFISH_URL)
    sh("tar", "-xf", "/tmp/sf/sf.tar", "-C", "/tmp/sf")
    ENGINE = next(f for f in glob.glob("/tmp/sf/**/stockfish*", recursive=True) if os.path.isfile(f) and not f.endswith((".tar", ".txt", ".md")) and os.access(f, os.X_OK))
    sh("sh", "-c", f"echo quit | {ENGINE} | head -1 | tee {OUT}/stockfish_version.txt")
    sh("sha256sum", "/tmp/sf/sf.tar")
    for name, path in {"human": HUMAN, **GENERATED}.items():
        sh(*CLI, "humanlike-quality", "--games", path, "--engine", ENGINE, "--workers", str(WORKERS), "--max-moves", MAX_MOVES or "40", "--out", f"{OUT}/quality_{name}.jsonl")
    EXTRA = ["--quality", *[f"{n}={OUT}/quality_{n}.jsonl" for n in ("human", *GENERATED)]] + (["--max-moves", MAX_MOVES] if MAX_MOVES else [])


## The test

In [ ]:
sh(*CLI, "humanlike-c2st", "--human", HUMAN, "--games", *[f"{k}={v}" for k, v in GENERATED.items()], "--folds", "5", "--boot", "1000", *EXTRA, "--out", f"{OUT}/c2st{'_engine' if ENGINE_FEATURES else ''}.md")